# Part 4: The ”Budget Keeper” (Token Economics) (15 Points)
- The Problem: Users are forwarding long chain-spam messages.
- Your Task:
    - Use uLls.token_uLls.count_messages_tokens.
    - Logic: If message > 150 tokens → Truncate or Summarize (overflow_summarize.v1) before processing.
- Success Check: Print ”BLOCKED/TRUNCATED” for a spam input.

In [1]:
# Import libraries
import sys
sys.path.append('..')

from utils.token_utils import pick_encoding, count_text_tokens
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
import tiktoken
import pandas as pd
from IPython.display import Markdown, display

In [103]:
# pick a model
model = pick_model('groq', 'general')
print(f'Using model: {model}')

client_capped = LLMClient('groq', model)

MAX_TOKENS = 150


Using model: llama-3.1-8b-instant


In [115]:
def analyze__overflow_and_spam_context(context: str):
    context_token_count = count_text_tokens(context, 'groq', model)
    out = None

    if context_token_count > MAX_TOKENS:
        print(f"Context exceeded limit of {MAX_TOKENS} ({context_token_count} tokens). Summarizing...")
        prompt_text, spec = render(
            'overflow_summarize.v1',
            context=context,
            max_tokens_context='150',
            task='Identify if the message is chain spam. If so, add "BLOCK Chain-Spam" to the summary of crisis-related information. Otherwise, return only essential crisis-related information. ',
            format='plain text summary only with no markdown syntax, labels, step numbers or extra commentary'
        )
        
        response = client_capped.chat([{'role': 'user', 'content': prompt_text}], temperature=0.2, max_tokens=MAX_TOKENS)
        log_llm_call('groq', model, 'overflow_summarize.v1', response['latency_ms'], response['usage'])
        summarized_context = response['text'].strip()

        out = {
            'original': context,
            'summarized': summarized_context,
            'original_token_count': context_token_count,
            'summarized_token_count': count_text_tokens(summarized_context, 'groq', model),
            'overflow_handled': True,
            'raw_response': response
        }
        if 'block chain-spam' in out['summarized'].lower():
            print("BLOCKED/TRUNCATED")

    else:
        print(f"Context is within the token limit ({context_token_count} tokens). No summarization needed.")

        out = {
            'original': context,
            'summarized': None,
            'original_token_count': context_token_count,
            'summarized_token_count': None,
            'overflow_handled': False,
            'raw_response': None
        }
    
    for key, value in out.items():
        print(f"{key}: {value}")
    
    return out

In [117]:
# contains test cases for valid but long context, spam, and acceptable context
test_cases = ['''
We are stranded on the second floor of a flooded house near the old bridge in Kaduwela. The water rose quickly after the river overflowed 
and now the ground floor is completely submerged. There are six of us here, including my elderly parents, my sister, and two small children 
aged four and seven. The current outside is strong and debris is floating past, including tree branches and parts of roofs. We tried to move 
to higher ground earlier but the road disappeared under water within minutes. My father has difficulty breathing and his inhaler is almost 
finished. We have very little drinking water left and no dry food. Mobile signal is weak, so I am sending this message hoping it reaches rescue 
teams. The roof is accessible but slippery, and we are afraid to stay there because of the rain and wind. Nightfall is approaching and visibility 
is getting worse. Nearby houses appear empty and we cannot see any boats in this area. Please send a rescue boat urgently. We can wave a white 
cloth from the balcony. Coordinates approximately near the temple junction, yellow two-story house with blue gate.
''',
'''
Please forward this message to every family and school group right now. A contact said there will be a citywide service interruption tonight, 
and only people who share this warning will be prepared. Do not wait for official updates, because they are always delayed. Keep repeating this 
text in all groups so nobody misses it. The message says everyone must send it to ten contacts in five minutes, then ask each contact to do the 
same. It also says screenshots are not enough; copy the exact wording and repost it again after one hour. No source, hotline, or timestamp is 
included, but it uses urgent language, capital letters, and emotional pressure. It claims that ignoring the message could put neighbors at risk, 
and that forwarding proves you care. It tells readers to trust friends more than verified channels and to treat silence as confirmation. This is 
a classic chain-forward pattern designed to spread quickly through fear and repetition, not evidence. The instruction is simple: forward now, keep 
forwarding, and do not question the content until tomorrow morning. After reading, people are asked to add prayer emojis, mark the text as urgent, 
and repost hourly so the rumor stays visible overnight.
''',
'SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.'
]

In [118]:
results = []
for idx, test in enumerate(test_cases):
    print(f"\n--- Analyzing Test Case {idx+1} ---")
    result = analyze__overflow_and_spam_context(test)
    results.append(result)


--- Analyzing Test Case 1 ---
Context exceeded limit of 150 (225 tokens). Summarizing...
original: 
We are stranded on the second floor of a flooded house near the old bridge in Kaduwela. The water rose quickly after the river overflowed 
and now the ground floor is completely submerged. There are six of us here, including my elderly parents, my sister, and two small children 
aged four and seven. The current outside is strong and debris is floating past, including tree branches and parts of roofs. We tried to move 
to higher ground earlier but the road disappeared under water within minutes. My father has difficulty breathing and his inhaler is almost 
finished. We have very little drinking water left and no dry food. Mobile signal is weak, so I am sending this message hoping it reaches rescue 
teams. The roof is accessible but slippery, and we are afraid to stay there because of the rain and wind. Nightfall is approaching and visibility 
is getting worse. Nearby houses appear empty 

In [119]:
result_df = pd.DataFrame(results)
result_df['saved_tokens'] = result_df.apply(lambda row: row['original_token_count'] - row['summarized_token_count'] if row['summarized_token_count'] is not None else 0, axis=1)
result_df['status'] = result_df.apply(lambda row: 'BLOCKED/TRUNCATED' if row['summarized'] is not None and 'block chain-spam' in str(row['summarized']).lower() else row['summarized'], axis=1)

result_df

,original,summarized,original_token_count,summarized_token_count,overflow_handled,raw_response,saved_tokens,status
0,\nWe are stranded on the second floor of a flo...,We are stranded on the second floor of a flood...,225,139.0,True,{'text': 'We are stranded on the second floor ...,86.0,We are stranded on the second floor of a flood...
1,\nPlease forward this message to every family ...,There will be a citywide service interruption ...,241,102.0,True,{'text': 'There will be a citywide service int...,139.0,BLOCKED/TRUNCATED
2,SOS: 5 people trapped on a roof in Ja-Ela (Gam...,NaN,26,NaN,False,None,NaN,NaN
